In [ ]:
# ============================================================
# TALTECH AI PIPELINE
# NOTEBOOK 07C — GRAPH REPRESENTATION / OPERATIONAL TOPOLOGY
# Portable Windows/Linux
#
# Scientific purpose:
# - 7 device UUIDs are graph nodes.
# - NO UUID -> IfcSpace relation is invented.
# - Operational inter-device edges are inferred ONLY from TRAIN.
# - Similarity uses aligned multivariate T/RH/CO2 training trajectories.
# - A symmetric k-NN graph (k=2) avoids choosing a post-hoc threshold.
# - A shuffled graph with the same node/edge count is generated as control.
# - G_native contains nodes only (no inferred inter-device edges).
# - Graph snapshots use the same 60-min history and t+15 target.
# - Snapshot eligibility is audited against the 07A common sample keys.
#
# IMPORTANT:
# This notebook does NOT train the GCN.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
OUTPUTS = PROJECT_DIR / "outputs"
COMMON_FILE = OUTPUTS / "06_common_experiment" / "01_common_supervised_dataset.csv"
TABULAR_FILE = OUTPUTS / "07A_tabular" / "01_tabular_all_unscaled.csv"
OUT = OUTPUTS / "07C_graph"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["temperature", "relative_humidity", "co2"]
TARGET = "target_temperature_t_plus_15"
STEP_MIN = 15
SEQ_LEN = 5
K_NEIGHBORS = 2
RANDOM_SEED = 42

print("=" * 66)
print("NOTEBOOK 07C — GRAPH REPRESENTATION / OPERATIONAL TOPOLOGY")
print("=" * 66)
print("Common input :", COMMON_FILE)
print("07A reference:", TABULAR_FILE)
print("Output       :", OUT)

assert COMMON_FILE.exists(), f"Missing {COMMON_FILE}"
assert TABULAR_FILE.exists(), f"Missing {TABULAR_FILE}"

# ------------------------------------------------------------
# 1. LOAD DATA AND 07A REFERENCE KEYS
# ------------------------------------------------------------
df = pd.read_csv(COMMON_FILE)
tab = pd.read_csv(TABULAR_FILE)

for x in (df, tab):
    x["timestamp"] = pd.to_datetime(x["timestamp"], utc=True)
    if "target_timestamp" in x.columns:
        x["target_timestamp"] = pd.to_datetime(x["target_timestamp"], utc=True)

for c in FEATURES + [TARGET]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

devices = sorted(df["uuid"].unique())
N = len(devices)
assert N == 7, f"Expected 7 devices, found {N}"
node_index = {u: i for i, u in enumerate(devices)}

pd.DataFrame({
    "node_index": range(N),
    "uuid": devices
}).to_csv(OUT / "01_graph_node_index.csv", index=False)

# ------------------------------------------------------------
# 2. TRAIN-ONLY MULTIVARIATE OPERATIONAL SIMILARITY
# ------------------------------------------------------------
# Only timestamps at which ALL 7 devices have T/RH/CO2 in TRAIN
# are used to infer topology.
train = df[df["split"] == "train"].copy()

wide_parts = []
for var in FEATURES:
    p = train.pivot_table(
        index="timestamp", columns="uuid", values=var, aggfunc="mean"
    ).reindex(columns=devices)
    p.columns = pd.MultiIndex.from_product([[var], p.columns])
    wide_parts.append(p)

wide = pd.concat(wide_parts, axis=1).sort_index()
complete_train_times = wide.dropna().index
wide = wide.loc[complete_train_times]

assert len(wide) > 0, "No complete 7-device training timestamps."

# Standardize each device-variable trajectory using TRAIN only.
# Then concatenate the three trajectories per device.
vectors = {}
for u in devices:
    cols = [(v, u) for v in FEATURES]
    arr = wide.loc[:, cols].to_numpy(dtype=float)
    mean = arr.mean(axis=0, keepdims=True)
    std = arr.std(axis=0, keepdims=True)
    std[std == 0] = 1.0
    z = (arr - mean) / std
    vectors[u] = z.reshape(-1)

# Pearson correlation of standardized multivariate trajectories.
sim = np.eye(N, dtype=float)
for i in range(N):
    for j in range(i + 1, N):
        r = np.corrcoef(vectors[devices[i]], vectors[devices[j]])[0, 1]
        if not np.isfinite(r):
            r = 0.0
        sim[i, j] = sim[j, i] = r

sim_df = pd.DataFrame(sim, index=devices, columns=devices)
sim_df.to_csv(OUT / "02_train_operational_similarity_matrix.csv")

# Long-form similarity table
pairs = []
for i in range(N):
    for j in range(i + 1, N):
        pairs.append({
            "source_uuid": devices[i],
            "target_uuid": devices[j],
            "pearson_similarity": sim[i, j]
        })
pd.DataFrame(pairs).to_csv(
    OUT / "03_train_pairwise_similarity.csv", index=False
)

# ------------------------------------------------------------
# 3. BUILD SYMMETRIC k-NN OPERATIONAL GRAPH
# ------------------------------------------------------------
# Each node selects its k highest-similarity OTHER nodes.
# Union of directed selections -> undirected graph.
# This is an operational/statistical relation, NOT spatial relation.

edge_set = set()
selection_rows = []

for i, u in enumerate(devices):
    candidates = [(j, sim[i, j]) for j in range(N) if j != i]
    candidates.sort(key=lambda x: x[1], reverse=True)
    chosen = candidates[:K_NEIGHBORS]

    for rank, (j, score) in enumerate(chosen, start=1):
        v = devices[j]
        a, b = sorted((i, j))
        edge_set.add((a, b))
        selection_rows.append({
            "source_uuid": u,
            "selected_neighbor_uuid": v,
            "rank": rank,
            "train_similarity": score
        })

pd.DataFrame(selection_rows).to_csv(
    OUT / "04_knn_neighbor_selections.csv", index=False
)

operational_edges = []
for i, j in sorted(edge_set):
    operational_edges.append({
        "source_index": i,
        "target_index": j,
        "source_uuid": devices[i],
        "target_uuid": devices[j],
        "train_similarity": sim[i, j],
        "relation": "operational_similarity"
    })

op_edges_df = pd.DataFrame(operational_edges)
op_edges_df.to_csv(OUT / "05_operational_graph_edges.csv", index=False)

# Native graph: deliberately no inferred device-device edges.
native_edges_df = pd.DataFrame(columns=[
    "source_index", "target_index", "source_uuid",
    "target_uuid", "relation"
])
native_edges_df.to_csv(OUT / "06_native_graph_interdevice_edges.csv", index=False)

# ------------------------------------------------------------
# 4. SHUFFLED CONTROL GRAPH — SAME NODE COUNT + EDGE COUNT
# ------------------------------------------------------------
# Random simple undirected graph with exactly the same number of edges.
# It is NOT claimed to preserve degree sequence.
rng = np.random.default_rng(RANDOM_SEED)
all_possible = [(i, j) for i in range(N) for j in range(i + 1, N)]
m = len(edge_set)

op_edge_set = set(edge_set)
shuffled_set = None

for _ in range(10000):
    ids = rng.choice(len(all_possible), size=m, replace=False)
    candidate = {all_possible[k] for k in ids}
    if candidate != op_edge_set:
        shuffled_set = candidate
        break

assert shuffled_set is not None

shuffled_rows = [{
    "source_index": i,
    "target_index": j,
    "source_uuid": devices[i],
    "target_uuid": devices[j],
    "relation": "shuffled_control"
} for i, j in sorted(shuffled_set)]

pd.DataFrame(shuffled_rows).to_csv(
    OUT / "07_shuffled_graph_edges.csv", index=False
)

# Save edge_index arrays in PyTorch-Geometric-style directed format.
def undirected_edge_index(edges):
    directed = []
    for i, j in sorted(edges):
        directed.extend([(i, j), (j, i)])
    if not directed:
        return np.empty((2, 0), dtype=np.int64)
    return np.asarray(directed, dtype=np.int64).T

np.save(OUT / "edge_index_operational.npy",
        undirected_edge_index(edge_set))
np.save(OUT / "edge_index_native.npy",
        np.empty((2, 0), dtype=np.int64))
np.save(OUT / "edge_index_shuffled.npy",
        undirected_edge_index(shuffled_set))

# ------------------------------------------------------------
# 5. GRAPH SNAPSHOT ELIGIBILITY ON EXACT 07A SAMPLE KEYS
# ------------------------------------------------------------
# A standard full-graph GCN snapshot predicts all 7 device nodes
# at one timestamp. Therefore a timestamp is fully comparable only
# if all 7 (UUID,timestamp) keys are present in 07A.

tab_keys = tab[["uuid", "timestamp", "split", TARGET]].copy()

counts = (
    tab_keys.groupby(["split", "timestamp"])["uuid"]
    .nunique()
    .rename("devices_available")
    .reset_index()
)

eligible_times = counts[counts["devices_available"] == N].copy()

eligibility_audit = (
    counts.groupby("split")
    .agg(
        candidate_timestamps=("timestamp", "size"),
        complete_7node_timestamps=("devices_available",
                                   lambda s: int((s == N).sum())),
        min_devices=("devices_available", "min"),
        max_devices=("devices_available", "max")
    )
    .reset_index()
)
eligibility_audit["excluded_timestamps"] = (
    eligibility_audit["candidate_timestamps"]
    - eligibility_audit["complete_7node_timestamps"]
)
eligibility_audit["complete_pct"] = (
    100 * eligibility_audit["complete_7node_timestamps"]
    / eligibility_audit["candidate_timestamps"]
)
eligibility_audit.to_csv(
    OUT / "08_graph_snapshot_eligibility_audit.csv", index=False
)

# ------------------------------------------------------------
# 6. BUILD 5x7x3 GRAPH SEQUENCES FOR ELIGIBLE ENDPOINTS
# ------------------------------------------------------------
# X_graph sample shape = [5 time steps, 7 nodes, 3 features]
# y_graph sample shape = [7 nodes]
#
# We require every history timestamp and endpoint to be inside the
# same split and to have all 7 devices.

# Fast lookup from common data
lookup = df.set_index(["split", "timestamp", "uuid"]).sort_index()

X_graph, y_graph, graph_meta = [], [], []

for row in eligible_times.itertuples(index=False):
    split = row.split
    end_t = row.timestamp
    history_times = [
        end_t - pd.Timedelta(minutes=STEP_MIN * k)
        for k in range(SEQ_LEN - 1, -1, -1)
    ]

    sample = np.empty((SEQ_LEN, N, len(FEATURES)), dtype=np.float32)
    targets = np.empty(N, dtype=np.float32)
    valid = True

    for ti, t in enumerate(history_times):
        for ni, u in enumerate(devices):
            key = (split, t, u)
            if key not in lookup.index:
                valid = False
                break
            r = lookup.loc[key]
            # duplicate safety
            if isinstance(r, pd.DataFrame):
                valid = False
                break
            vals = r[FEATURES].to_numpy(dtype=float)
            if not np.isfinite(vals).all():
                valid = False
                break
            sample[ti, ni, :] = vals
        if not valid:
            break

    if not valid:
        continue

    # Targets at endpoint must match 07A keys and common target.
    for ni, u in enumerate(devices):
        key = (split, end_t, u)
        r = lookup.loc[key]
        targets[ni] = float(r[TARGET])

    X_graph.append(sample)
    y_graph.append(targets)
    graph_meta.append({
        "split": split,
        "sequence_start": history_times[0],
        "sequence_end": end_t,
        "target_timestamp": end_t + pd.Timedelta(minutes=STEP_MIN)
    })

assert X_graph, "No valid full graph sequences."

X_graph = np.stack(X_graph)
y_graph = np.stack(y_graph)
graph_meta = pd.DataFrame(graph_meta)

# ------------------------------------------------------------
# 7. TRAIN-ONLY SCALING FOR GRAPH NODE FEATURES
# ------------------------------------------------------------
# One mean/std per physical variable, fitted only on graph TRAIN.
train_idx = np.flatnonzero(graph_meta["split"].to_numpy() == "train")
assert len(train_idx) > 0

train_flat = X_graph[train_idx].reshape(-1, len(FEATURES))
mean = train_flat.mean(axis=0)
std = train_flat.std(axis=0)
std[std == 0] = 1.0

X_graph_scaled = ((X_graph - mean) / std).astype(np.float32)

pd.DataFrame({
    "feature": FEATURES,
    "train_mean": mean,
    "train_scale": std
}).to_csv(OUT / "09_graph_scaler_train_statistics.csv", index=False)

# ------------------------------------------------------------
# 8. SAVE GRAPH TENSORS
# ------------------------------------------------------------
split_order = ["train", "validation", "test"]
snapshot_audit = []

for s in split_order:
    ids = np.flatnonzero(graph_meta["split"].to_numpy() == s)
    part = graph_meta.iloc[ids].reset_index(drop=True)

    np.save(OUT / f"X_graph_{s}_unscaled.npy", X_graph[ids])
    np.save(OUT / f"X_graph_{s}_scaled.npy", X_graph_scaled[ids])
    np.save(OUT / f"y_graph_{s}.npy", y_graph[ids])
    part.to_csv(OUT / f"metadata_graph_{s}.csv", index=False)

    snapshot_audit.append({
        "split": s,
        "graph_snapshots": len(ids),
        "node_level_predictions": len(ids) * N,
        "X_shape": str(tuple(X_graph[ids].shape)),
        "y_shape": str(tuple(y_graph[ids].shape)),
        "start": part["sequence_end"].min() if len(part) else None,
        "end": part["sequence_end"].max() if len(part) else None
    })

snapshot_audit = pd.DataFrame(snapshot_audit)
snapshot_audit.to_csv(
    OUT / "10_graph_tensor_audit.csv", index=False
)
graph_meta.to_csv(
    OUT / "11_all_graph_snapshot_metadata.csv", index=False
)

# ------------------------------------------------------------
# 9. PROTOCOL MANIFEST
# ------------------------------------------------------------
protocol = {
    "representation": "graph",
    "nodes": N,
    "node_identity": "operational device UUID",
    "spatial_uuid_ifc_mapping_used": False,
    "spatial_claim": False,
    "native_interdevice_edges": 0,
    "operational_topology_source": "training data only",
    "operational_similarity":
        "Pearson correlation of concatenated train-standardized T/RH/CO2 trajectories",
    "operational_graph_rule": f"symmetric union of kNN selections, k={K_NEIGHBORS}",
    "threshold_tuned_on_validation_or_test": False,
    "shuffled_control_same_node_count": True,
    "shuffled_control_same_edge_count": True,
    "shuffled_control_degree_preserving": False,
    "sequence_length": SEQ_LEN,
    "history_minutes": 60,
    "time_step_minutes": 15,
    "node_features_per_step": FEATURES,
    "target": "temperature at t+15 min for each node",
    "scaling": "train graph snapshots only",
    "intended_ablation": [
        "G_native (no inferred inter-device edges)",
        "G_operational (train-only operational similarity)",
        "G_shuffled (random connectivity control)"
    ]
}
with (OUT / "12_graph_protocol.json").open("w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)

# ------------------------------------------------------------
# 10. FINAL REPORT
# ------------------------------------------------------------
print("\n" + "=" * 66)
print("TRAIN-ONLY TOPOLOGY")
print("=" * 66)
print("Complete 7-device TRAIN timestamps used for similarity:",
      len(complete_train_times))
print("Nodes              :", N)
print("k                  :", K_NEIGHBORS)
print("Operational edges  :", len(edge_set))
print("Native device edges:", 0)
print("Shuffled edges     :", len(shuffled_set))

print("\nOperational edges:")
print(op_edges_df.to_string(index=False))

print("\n" + "=" * 66)
print("07A -> FULL GRAPH SNAPSHOT ELIGIBILITY")
print("=" * 66)
print(eligibility_audit.to_string(index=False))

print("\n" + "=" * 66)
print("FINAL GRAPH TENSORS")
print("=" * 66)
print(snapshot_audit.to_string(index=False))

print("\nVALIDATION")
print("✓ No UUID -> IfcSpace relation invented")
print("✓ Operational topology inferred from TRAIN only")
print("✓ No validation/test information used for edge construction")
print("✓ Operational relation is explicitly statistical, not spatial")
print("✓ Native graph contains no inferred inter-device edges")
print("✓ Shuffled control has same nodes and edge count")
print("✓ Graph history = 60 min, matching 07A/07B")
print("✓ Graph target = T(t+15) for each device")
print("✓ Graph snapshot eligibility explicitly audited")
print("\nNOTE: Review 08_graph_snapshot_eligibility_audit.csv before GCN training.")
